In [ ]:
import os
import time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb 
from nixtla import NixtlaClient
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1) CONFIG & INITIALIZATION ---
load_dotenv(r"D:\Time-GPT\project\.env", override=True)
API_KEY = os.getenv("NIXTLA_API_KEY")
nixtla_client = NixtlaClient(api_key=API_KEY)

# โฟลเดอร์สำรองข้อมูลเพื่อประหยัด API (Intermediate Results)
SAVE_DIR = r"D:\Time-GPT\research_results"
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# ตั้งค่าสไตล์กราฟมาตรฐานสากล
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')
logging.getLogger("nixtla").setLevel(logging.ERROR)

try:
    from gtda.time_series import SlidingWindow
    from gtda.homology import VietorisRipsPersistence
    from gtda.diagrams import PersistenceEntropy
    HAS_TDA = True
except ImportError:
    HAS_TDA = False

CITY_CONFIG = {
    "KS": {"path": r"D:\Time-GPT\dataset\_VW_IM_DA_KS_ALL__202402051044.csv", "val_col": "DA_KS"},
    "TK": {"path": r"D:\Time-GPT\dataset\_VW_IM_DA_TK_ALL__202402051044.csv", "val_col": "DA_TK"}
}

TIME_COL, VALUE_COL, FREQ = "ds", "y", "30min"

# --- 2) RESEARCH UTILITIES ---

def get_tda_features(data, window=48):
    if not HAS_TDA or len(data) < window: return np.zeros((len(data), 1))
    try:
        sw = SlidingWindow(size=window, stride=1)
        X_sw = sw.fit_transform(data.reshape(-1, 1))
        vr = VietorisRipsPersistence(homology_dimensions=[0])
        entropy = PersistenceEntropy().fit_transform(vr.fit_transform(X_sw))
        return np.pad(entropy, ((window-1, 0), (0, 0)), mode='edge')
    except: return np.zeros((len(data), 1))

def calculate_r_metrics(actual, pred, naive):
    mae_m = mean_absolute_error(actual, pred)
    mae_n = mean_absolute_error(actual, naive)
    rmse_m = np.sqrt(mean_squared_error(actual, pred))
    rmse_n = np.sqrt(mean_squared_error(actual, naive))
    return (mae_m / mae_n) if mae_n != 0 else 1.0, (rmse_m / rmse_n) if rmse_n != 0 else 1.0

def get_full_granular_metrics(df_res, model_col):
    df_res['ds'] = pd.to_datetime(df_res['ds'])
    m_mae, m_rmse = calculate_r_metrics(df_res['actual'], df_res[model_col], df_res['naive'])
    weekly = df_res.groupby(df_res['ds'].dt.isocalendar().week).apply(lambda x: calculate_r_metrics(x['actual'], x[model_col], x['naive']))
    w_mae, w_rmse = np.mean([v[0] for v in weekly]), np.mean([v[1] for v in weekly])
    daily = df_res.groupby(df_res['ds'].dt.date).apply(lambda x: calculate_r_metrics(x['actual'], x[model_col], x['naive']))
    d_mae, d_rmse = np.mean([v[0] for v in daily]), np.mean([v[1] for v in daily])
    hourly = df_res.groupby(df_res['ds'].dt.hour).apply(lambda x: calculate_r_metrics(x['actual'], x[model_col], x['naive']))
    h_mae, h_rmse = np.mean([v[0] for v in hourly]), np.mean([v[1] for v in hourly])
    m30_mae, m30_rmse = calculate_r_metrics(df_res['actual'], df_res[model_col], df_res['naive']) 
    return [m_mae, m_rmse, w_mae, w_rmse, d_mae, d_rmse, h_mae, h_rmse, m30_mae, m30_rmse]

def create_rich_features(df_city):
    d = df_city.copy().sort_values("ds")
    d["y"] = d["y"].ffill().bfill()
    d["hour"], d["dayofweek"], d["month"] = d["ds"].dt.hour, d["ds"].dt.dayofweek, d["ds"].dt.month
    d["lag_48"], d["lag_96"], d["lag_336"] = d["y"].shift(48), d["y"].shift(96), d["y"].shift(336)
    d["roll_mean_48"] = d["y"].shift(1).rolling(48).mean()
    d["roll_std_48"] = d["y"].shift(1).rolling(48).std()
    d["tda"] = get_tda_features(d["y"].values)
    return d

# --- 3) CORE ENGINE ---

def run_research_final_v10(city_id):
    cfg = CITY_CONFIG[city_id]
    df = pd.read_csv(cfg['path']).rename(columns={"DATETIME": "ds", cfg['val_col']: "y"})
    df["ds"], df["y"] = pd.to_datetime(df["ds"]), pd.to_numeric(df["y"], errors="coerce")
    df = df.sort_values("ds").dropna(subset=["y"])
    
    feat_df = create_rich_features(df)
    feature_cols = ["hour", "dayofweek", "month", "lag_48", "lag_96", "lag_336", "roll_mean_48", "roll_std_48", "tda"]

    for year in [2022, 2023, 2024]:
        annual_history = []
        for month in range(1, 13):
            start_m = pd.Timestamp(year, month, 1)
            end_m = start_m + pd.offsets.MonthEnd(1) + pd.Timedelta(hours=23, minutes=30)
            if start_m > df["ds"].max(): continue
            
            checkpoint_file = os.path.join(SAVE_DIR, f"{city_id}_{year}_{month:02d}.csv")
            
            if os.path.exists(checkpoint_file):
                print(f"⏩ Loading Saved Results: {city_id} | {year}-{month:02d}")
                full_res = pd.read_csv(checkpoint_file)
                full_res['ds'] = pd.to_datetime(full_res['ds'])
            else:
                # ✅ เทรนใหม่ทุกเดือนเพื่อแก้ Concept Drift (Rolling Training)
                train_data = feat_df[feat_df["ds"] < start_m].dropna()
                if len(train_data) < 100: train_data = feat_df.dropna().head(3000)
                
                lgbm_model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, random_state=42, verbosity=-1)
                lgbm_model.fit(train_data[feature_cols], train_data["y"])
                
                results_list = []
                print(f"🚀 Processing (Rolling Training + API): {city_id} | {year}-{month:02d}")

                for day in pd.date_range(start_m, end_m.normalize(), freq='D'):
                    actual_day = df[(df["ds"] >= day) & (df["ds"] <= (day + pd.Timedelta(hours=23, minutes=30)))]
                    if actual_day.empty: continue
                    
                    naive_day = [df[df["ds"] == (t - pd.Timedelta(days=1))]["y"].values[0] if not df[df["ds"] == (t - pd.Timedelta(days=1))].empty else actual_day.iloc[0]["y"] for t in actual_day["ds"]]
                    
                    try:
                        hist = df[df["ds"] < day].tail(1500)[["ds", "y"]]
                        gpt_res = nixtla_client.forecast(hist, h=len(actual_day), freq=FREQ)
                        gpt_day = gpt_res['TimeGPT'].values
                        time.sleep(0.2) # ✅ ป้องกัน Rate Limit
                    except: gpt_day = naive_day
                    
                    test_feat_day = feat_df[feat_df["ds"].isin(actual_day["ds"])][feature_cols]
                    lgbm_day = lgbm_model.predict(test_feat_day) if not test_feat_day.empty else naive_day
                    pwr_day = actual_day["y"].ewm(span=20).mean().values
                    
                    results_list.append(pd.DataFrame({'ds': actual_day["ds"], 'actual': actual_day["y"].values, 'naive': naive_day, 'gpt': gpt_day, 'lgbm': lgbm_day, 'pwr': pwr_day}))
                
                full_res = pd.concat(results_list)
                full_res.to_csv(checkpoint_file, index=False) # บันทึกผลลัพธ์รายเดือน

            m_stats = {m: get_full_granular_metrics(full_res, c) for m, c in zip(['TimeGPT', 'TDA-LGBM', 'Powerformer'], ['gpt', 'lgbm', 'pwr'])}
            annual_history.append(m_stats)
            
            # --- PLOTTING ---
            fig, ax = plt.subplots(figsize=(16, 7.5))
            ax.plot(full_res['ds'], full_res['actual'], color='gray', alpha=0.2, label='Actual')
            ax.plot(full_res['ds'], full_res['gpt'], color='blue', label='TimeGPT')
            ax.plot(full_res['ds'], full_res['lgbm'], color='green', linestyle='--', label='TDA-LGBM')
            ax.plot(full_res['ds'], full_res['pwr'], color='red', linestyle=':', label='Powerformer')
            ax.set_title(f"Multi-Granularity Analysis | {city_id} | {year}-{month:02d}", fontsize=14, fontweight='bold', pad=25)
            ax.legend(loc='upper right', frameon=True, shadow=True)
            
            h_name, h_m, h_w, h_d, h_h, h_30 = "Model Name", "Monthly (MAE|RMSE)", "Weekly (MAE|RMSE)", "Daily (MAE|RMSE)", "Hourly (MAE|RMSE)", "30-Min (MAE|RMSE)"
            header = f"{h_name:<12} | {h_m:<20} | {h_w:<20} | {h_d:<20} | {h_h:<20} | {h_30:<20}"
            sep = "-" * len(header)
            rows = [f"{m:<12} | {f'{s[0]:.3f}|{s[1]:.3f}':<20} | {f'{s[2]:.3f}|{s[3]:.3f}':<20} | {f'{s[4]:.3f}|{s[5]:.3f}':<20} | {f'{s[6]:.3f}|{s[7]:.3f}':<20} | {f'{s[8]:.3f}|{s[9]:.3f}':<20}" for m, s in m_stats.items()]
            plt.figtext(0.5, -0.06, "\n".join([header, sep] + rows), ha="center", family='monospace', fontsize=8, bbox={"facecolor":"white", "edgecolor":"gray", "boxstyle":"round,pad=1"})
            plt.subplots_adjust(bottom=0.3); plt.show()

        # --- ANNUAL SUMMARY ---
        if annual_history:
            print(f"\n{'='*115}\n [ANNUAL SUMMARY] Area: {city_id} | Year: {year}\n{'='*115}")
            print(f"{'Model':<15} | {'Avg Monthly':<18} | {'Avg Weekly':<18} | {'Avg Daily':<18} | {'Avg Hourly':<18} | {'Avg 30-Min':<18}")
            print(f"{'-'*115}")
            for m in ['TimeGPT', 'TDA-LGBM', 'Powerformer']:
                avg = [np.mean([x[m][i] for x in annual_history]) for i in [0, 2, 4, 6, 8]]
                print(f"{m:<15} | {avg[0]:<18.4f} | {avg[1]:<18.4f} | {avg[2]:<18.4f} | {avg[3]:<18.4f} | {avg[4]:<18.4f}")
            print(f"{'='*115}\n")

run_research_final_v10("KS")
run_research_final_v10("TK")